In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.decomposition import PCA
import scipy.sparse as sp

print("Imports done!")

Imports done!


In [ ]:
import os

os.makedirs('/content/data', exist_ok=True)

files = [
    'bio-decagon-combo.tar.gz',
    'bio-decagon-mono.tar.gz',
    'bio-decagon-targets.tar.gz',
    'bio-decagon-ppi.tar.gz'
]

base_url = 'http://snap.stanford.edu/decagon'

for f in files:
    out = f'/content/data/{f}'
    if not os.path.exists(out.replace('.tar.gz', '.csv')):
        print(f"Downloading {f}...")
        os.system(f'wget -q {base_url}/{f} -O {out}')
        os.system(f'tar -xzf {out} -C /content/data/')
        print(f"Done: {f}")
    else:
        print(f"Already exists: {f}")

print("\nFiles in data folder:")
print([f for f in os.listdir('/content/data') if f.endswith('.csv')])

Already exists: bio-decagon-combo.tar.gz
Already exists: bio-decagon-mono.tar.gz
Already exists: bio-decagon-targets.tar.gz
Already exists: bio-decagon-ppi.tar.gz

Files in data folder:
['bio-decagon-combo.csv', '._bio-decagon-ppi.csv', '._bio-decagon-targets.csv', '._bio-decagon-mono.csv', 'bio-decagon-mono.csv', '._bio-decagon-combo.csv', 'bio-decagon-targets.csv', 'bio-decagon-ppi.csv']


In [ ]:
combo   = pd.read_csv('/content/data/bio-decagon-combo.csv')
targets = pd.read_csv('/content/data/bio-decagon-targets.csv')

drug1_col   = 'STITCH 1'
drug2_col   = 'STITCH 2'
se_col      = 'Polypharmacy Side Effect'
se_name_col = 'Side Effect Name'


In [ ]:
se_counts      = combo[se_col].value_counts()
valid_se       = se_counts[se_counts >= 500].index
combo_filtered = combo[combo[se_col].isin(valid_se)].copy()

top25_common = se_counts[se_counts >= 500].head(25).index
top25_rare   = se_counts[se_counts >= 500].tail(25).index
mixed_se     = top25_common.tolist() + top25_rare.tolist()

combo_filtered = combo_filtered[
    combo_filtered[se_col].isin(mixed_se)
].copy()

common_se = top25_common.tolist()
rare_se   = top25_rare.tolist()

all_drugs = pd.unique(combo_filtered[[drug1_col, drug2_col]].values.ravel())
drug2idx  = {d: i for i, d in enumerate(all_drugs)}
n_drugs   = len(all_drugs)

print(f"Common side effects (top 25): {len(common_se)}")
print(f"Rare side effects (bottom 25): {len(rare_se)}")
print(f"\nBottom 5 RARE side effects (should include dyshidrosis, sarcoma etc.):")
for se in top25_rare[-5:]:
    name  = combo[combo[se_col]==se][se_name_col].iloc[0]
    count = se_counts[se]
    print(f"  {name:<40} {count:,} pairs")

Common side effects (top 25): 25
Rare side effects (bottom 25): 25

Bottom 5 RARE side effects (should include dyshidrosis, sarcoma etc.):
  coccydynia                               508 pairs
  neonatal respiratory distress syndrome   507 pairs
  impetigo                                 507 pairs
  Collagen disease                         504 pairs
  sarcoma                                  502 pairs


In [ ]:
prot_targets = targets.copy()
all_prots    = prot_targets.iloc[:, 1].unique()
prot2idx     = {p: i for i, p in enumerate(all_prots)}
n_prots      = len(all_prots)

rows, cols = [], []
for _, row in prot_targets.iterrows():
    drug = row.iloc[0]
    prot = row.iloc[1]
    if drug in drug2idx and prot in prot2idx:
        rows.append(drug2idx[drug])
        cols.append(prot2idx[prot])

drug_prot_adj   = sp.csr_matrix((np.ones(len(rows)), (rows, cols)),
                                 shape=(n_drugs, n_prots))
drug_prot_dense = drug_prot_adj.toarray()

pca           = PCA(n_components=50, random_state=42)
drug_features = pca.fit_transform(drug_prot_dense)
print(f"Drug features shape: {drug_features.shape}")
print(f"Variance explained:  {pca.explained_variance_ratio_.sum():.1%}")

Drug features shape: (645, 50)
Variance explained:  98.0%


In [ ]:
def train_rf_for_se(se_code, se_name, combo_df, drug2idx,
                     drug_features, n_drugs, seed=42):
    se_df  = combo_df[combo_df[se_col] == se_code]
    d1_ids = se_df[drug1_col].map(drug2idx).dropna().astype(int).values
    d2_ids = se_df[drug2_col].map(drug2idx).dropna().astype(int).values

    valid = (d1_ids < n_drugs) & (d2_ids < n_drugs)
    d1_ids, d2_ids = d1_ids[valid], d2_ids[valid]

    if len(d1_ids) < 50:
        print(f"  Skipping {se_name} — too few pairs ({len(d1_ids)})")
        return None

    n_pos = len(d1_ids)
    idx   = np.arange(n_pos)
    np.random.seed(seed)
    np.random.shuffle(idx)

    n_val  = max(10, int(n_pos * 0.1))
    n_test = max(10, int(n_pos * 0.1))

    train_idx = idx[n_test + n_val:]
    test_idx  = idx[:n_test]

    pos_set = set(zip(d1_ids.tolist(), d2_ids.tolist()))

    def sample_negatives(n_needed, seed):
        np.random.seed(seed)
        neg_r, neg_c = [], []
        attempts = 0
        while len(neg_r) < n_needed and attempts < n_needed * 50:
            r = np.random.randint(0, n_drugs)
            c = np.random.randint(0, n_drugs)
            if r != c and (r, c) not in pos_set:
                neg_r.append(r)
                neg_c.append(c)
            attempts += 1
        return np.array(neg_r), np.array(neg_c)

    def build_pair_features(r_ids, c_ids):
        feat_a = drug_features[r_ids]
        feat_b = drug_features[c_ids]
        return np.concatenate([feat_a, feat_b, feat_a * feat_b], axis=1)

    tr_d1, tr_d2 = d1_ids[train_idx], d2_ids[train_idx]
    neg_r, neg_c = sample_negatives(len(train_idx), seed=seed)
    neg_r, neg_c = neg_r[:len(train_idx)], neg_c[:len(train_idx)]

    X_train = np.vstack([
        build_pair_features(tr_d1, tr_d2),
        build_pair_features(neg_r, neg_c)
    ])
    y_train = np.concatenate([np.ones(len(tr_d1)), np.zeros(len(neg_r))])

    te_d1, te_d2 = d1_ids[test_idx], d2_ids[test_idx]
    neg_tr, neg_tc = sample_negatives(len(test_idx), seed=999)
    neg_tr, neg_tc = neg_tr[:len(test_idx)], neg_tc[:len(test_idx)]

    X_test = np.vstack([
        build_pair_features(te_d1, te_d2),
        build_pair_features(neg_tr, neg_tc)
    ])
    y_test = np.concatenate([np.ones(len(te_d1)), np.zeros(len(neg_tr))])

    clf = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_leaf=3,
        random_state=seed,
        n_jobs=-1
    )
    clf.fit(X_train, y_train)

    y_prob = clf.predict_proba(X_test)[:, 1]
    test_auroc = roc_auc_score(y_test, y_prob)
    test_auprc = average_precision_score(y_test, y_prob)

    return {
        'se_name': se_name,
        'se_code': se_code,
        'n_pairs': n_pos,
        'auroc':   test_auroc,
        'auprc':   test_auprc
    }

print("train_rf_for_se defined!")

train_rf_for_se defined!


In [ ]:
rf_results_common = []
rf_results_rare   = []

print("\n── Common side effects (25) ──")
for se_code in common_se:
    se_name = combo_filtered[combo_filtered[se_col] == se_code][se_name_col].iloc[0]
    result  = train_rf_for_se(se_code, se_name, combo_filtered, drug2idx,
                               drug_features, n_drugs)
    if result:
        rf_results_common.append(result)
        print(f"  {se_name:<40} AUROC: {result['auroc']:.3f}  AUPRC: {result['auprc']:.3f}")

print("\n── Rare side effects (25) ──")
for se_code in rare_se:
    se_name = combo_filtered[combo_filtered[se_col] == se_code][se_name_col].iloc[0]
    result  = train_rf_for_se(se_code, se_name, combo_filtered, drug2idx,
                               drug_features, n_drugs)
    if result:
        rf_results_rare.append(result)
        print(f"  {se_name:<40} AUROC: {result['auroc']:.3f}  AUPRC: {result['auprc']:.3f}")


── Common side effects (25) ──
  arterial pressure NOS decreased          AUROC: 0.750  AUPRC: 0.735
  anaemia                                  AUROC: 0.749  AUPRC: 0.728
  Difficulty breathing                     AUROC: 0.747  AUPRC: 0.724
  nausea                                   AUROC: 0.755  AUPRC: 0.731
  neumonia                                 AUROC: 0.752  AUPRC: 0.729
  Fatigue                                  AUROC: 0.758  AUPRC: 0.731
  Pain                                     AUROC: 0.770  AUPRC: 0.744
  diarrhea                                 AUROC: 0.747  AUPRC: 0.722
  asthenia                                 AUROC: 0.749  AUPRC: 0.720
  emesis                                   AUROC: 0.754  AUPRC: 0.728
  edema extremities                        AUROC: 0.757  AUPRC: 0.728
  body temperature increased               AUROC: 0.735  AUPRC: 0.711
  pleural pain                             AUROC: 0.760  AUPRC: 0.736
  abdominal pain                           AUROC: 0.755  A

In [ ]:
common_aurocs = [r['auroc'] for r in rf_results_common]
common_auprcs = [r['auprc'] for r in rf_results_common]
rare_aurocs   = [r['auroc'] for r in rf_results_rare]
rare_auprcs   = [r['auprc'] for r in rf_results_rare]

print("\n" + "=" * 50)
print("RANDOM FOREST SUMMARY")
print("=" * 50)
print(f"{'Category':<20} {'AUROC':>8} {'AUPRC':>8}")
print("-" * 50)
print(f"{'Common SEs (25)':<20} {np.mean(common_aurocs):>8.4f} {np.mean(common_auprcs):>8.4f}")
print(f"{'Rare SEs (25)':<20} {np.mean(rare_aurocs):>8.4f} {np.mean(rare_auprcs):>8.4f}")
print(f"{'Overall (50)':<20} {np.mean(common_aurocs+rare_aurocs):>8.4f} {np.mean(common_auprcs+rare_auprcs):>8.4f}")


RANDOM FOREST SUMMARY
Category                AUROC    AUPRC
--------------------------------------------------
Common SEs (25)        0.7562   0.7314
Rare SEs (25)          0.8177   0.7955
Overall (50)           0.7870   0.7635
